In [1]:
import pandas as pd
from pathlib import Path
from scipy.stats import mannwhitneyu, kruskal

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)


# Ready-to-move vs under construction

ready_price = df.loc[
    df["status"] == "Ready to move",
    "price_lakh"
]

under_price = df.loc[
    df["status"] == "Under Construction",
    "price_lakh"
]

ready_rate = df.loc[
    df["status"] == "Ready to move",
    "rate_per_sqft"
]

under_rate = df.loc[
    df["status"] == "Under Construction",
    "rate_per_sqft"
]

price_status_test = mannwhitneyu(
    ready_price,
    under_price,
    alternative="two-sided"
)

rate_status_test = mannwhitneyu(
    ready_rate,
    under_rate,
    alternative="two-sided"
)


# RERA approved vs not approved

approved_price = df.loc[
    df["rera_approval"] == "Approved by RERA",
    "price_lakh"
]

not_approved_price = df.loc[
    df["rera_approval"] == "Not approved by RERA",
    "price_lakh"
]

approved_rate = df.loc[
    df["rera_approval"] == "Approved by RERA",
    "rate_per_sqft"
]

not_approved_rate = df.loc[
    df["rera_approval"] == "Not approved by RERA",
    "rate_per_sqft"
]

price_rera_test = mannwhitneyu(
    approved_price,
    not_approved_price,
    alternative="two-sided"
)

rate_rera_test = mannwhitneyu(
    approved_rate,
    not_approved_rate,
    alternative="two-sided"
)


# BHK comparison
# Use only clearly specified 1 to 5 BHK properties

bhk_groups = [
    df.loc[df["bhk_count"] == bhk, "price_lakh"]
    for bhk in [1, 2, 3, 4, 5]
]

bhk_test = kruskal(*bhk_groups)


# Statistical results

statistical_results = pd.DataFrame({
    "comparison": [
        "Ready to move vs Under Construction - Price",
        "Ready to move vs Under Construction - Rate per sqft",
        "RERA Approved vs Not approved - Price",
        "RERA Approved vs Not approved - Rate per sqft",
        "1 to 5 BHK - Price"
    ],
    "test": [
        "Mann-Whitney U",
        "Mann-Whitney U",
        "Mann-Whitney U",
        "Mann-Whitney U",
        "Kruskal-Wallis"
    ],
    "statistic": [
        price_status_test.statistic,
        rate_status_test.statistic,
        price_rera_test.statistic,
        rate_rera_test.statistic,
        bhk_test.statistic
    ],
    "p_value": [
        price_status_test.pvalue,
        rate_status_test.pvalue,
        price_rera_test.pvalue,
        rate_rera_test.pvalue,
        bhk_test.pvalue
    ]
})

statistical_results["p_value"] = statistical_results["p_value"].round(6)

print(statistical_results.to_string(index=False))

                                         comparison           test    statistic  p_value
        Ready to move vs Under Construction - Price Mann-Whitney U 2.091251e+07 0.160684
Ready to move vs Under Construction - Rate per sqft Mann-Whitney U 1.776430e+07 0.000000
              RERA Approved vs Not approved - Price Mann-Whitney U 2.349686e+07 0.081340
      RERA Approved vs Not approved - Rate per sqft Mann-Whitney U 2.729823e+07 0.000000
                                 1 to 5 BHK - Price Kruskal-Wallis 5.273147e+03 0.000000
